# U-Net A/B: uncorrected vs parallax-corrected INSAT-3DR input

Model A is trained on the original INSAT patches (uncorrected).
Model B is trained on the same patches after PyGeoParallax-IN parallax correction.

Everything else is the same for both:

- the network, loss, optimiser (Adam 1e-6), batch size (1) and seed (42), from the group's U-Net notebooks;
- the labels (GPM DPR precipitation type), the pixel mask and the train/test split.

**Before you start**

1. Upload the folder `unet_colab` to the top of your Google Drive (`MyDrive/unet_colab`).
2. Choose **Runtime → Change runtime type → GPU** (T4 is fine).

**Resuming**

Each epoch is saved to `MyDrive/unet_colab/runs/<task>_<model>/`. When Colab disconnects, reconnect and run all cells again: training continues from the last saved epoch, and finished runs are skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/unet_colab'
RUNS = f'{BASE}/runs'
DATA = '/content/data'            # local copy: much faster than reading from Drive
!nvidia-smi -L
!python -c "import tensorflow as tf; print('TensorFlow', tf.__version__, tf.config.list_physical_devices('GPU'))"

In [ ]:
# Copy the dataset to the Colab disk (about 1-2 min each session)
!mkdir -p {DATA}
!test -f {DATA}/train.npz || cp {BASE}/data/train.npz {DATA}/
!test -f {DATA}/test.npz  || cp {BASE}/data/test.npz  {DATA}/
!ls -lh {DATA}

## Quick test (run once, about 1-2 min)
This trains 1 epoch on 50 patches into a throw-away folder, to check that everything works and to time it.

In [ ]:
!python {BASE}/scripts/train_unet_ab.py --model B --task fourclass --data {DATA} --out /content/smoke --epochs 1 --max-train 50

## Train
Runs are trained one after another. Start with the four-class pair (the main result); the other two tasks follow.
Each run is 100 epochs. A finished run is skipped, so you can run this cell again after a disconnect.

In [ ]:
RUNS_TO_DO = [('fourclass', 'A'), ('fourclass', 'B'),
              ('stratconv', 'A'), ('stratconv', 'B'),
              ('rain', 'A'), ('rain', 'B')]
for task, model in RUNS_TO_DO:
    print(f'==================== {task} model {model} ====================')
    !python {BASE}/scripts/train_unet_ab.py --model {model} --task {task} --data {DATA} --out {RUNS}

## Compare A and B
Run when both runs of a task are finished. B − A with 95% intervals from a paired bootstrap over test scenes.

In [ ]:
import os
for task in ['fourclass', 'stratconv', 'rain']:
    if all(os.path.exists(f'{RUNS}/{task}_{m}/test_metrics.json') for m in 'AB'):
        !python {BASE}/scripts/compare_unet_ab.py --runs {RUNS} --task {task}
    else:
        print(f'{task}: not finished yet')

In [ ]:
# Learning curves
import json, matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, task in zip(axes, ['fourclass', 'stratconv', 'rain']):
    for m, c in (('A', '#888888'), ('B', '#2a78d6')):
        p = f'{RUNS}/{task}_{m}/history.json'
        if not os.path.exists(p):
            continue
        h = json.load(open(p))
        ax.plot(h['epoch'], h['loss'], color=c, label=f'{m} train')
        ve = [(e, v[0]) for e, v in zip(h['epoch'], h['val']) if v]
        if ve:
            ax.plot(*zip(*ve), 'o--', color=c, label=f'{m} val')
    ax.set_title(task); ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax.legend()
plt.tight_layout(); plt.savefig(f'{RUNS}/learning_curves.png', dpi=120); plt.show()